# Blend

This notebook **doesn't train anything**. It loads the predictions saved by your model notebooks, checks how well each one and each combination scores, and writes a blended submission.

**Run the model notebooks first:** `04_catboost`, `03_lightgbm` and `05_logreg`. Whenever you change a model, rerun its notebook and then this one.

## 1. Imports

`rankdata` turns scores into ranks: the lowest score becomes 1, the next 2, and so on.

## Project folders

This notebook lives in `notebooks/`, but the data, shared code and outputs are in other folders. This cell finds the project folder and imports their locations from `scripts/paths.py`:

| Name | Folder | Holds |
|---|---|---|
| `DATA` | `data/` | Competition files |
| `PREDS` | `preds_for_blend/` | Saved predictions for blending |
| `SUBMISSIONS` | `submissions/` | Files to upload |
| `LOGS` | `logs/` | CatBoost training logs |

It also adds `scripts/` to Python's search path, so `from features import ...` works.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from paths import DATA, LOGS, PREDS, SUBMISSIONS

In [86]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

## 2. Load the real answers and the saved predictions

We only need `y` to score the predictions, plus the test IDs for the submission.

The loop loads every model whose files exist and skips any you haven't run yet. The `assert` checks that each file has one prediction per row, which catches a file saved from an older or broken run.

In [ ]:
TARGET = "12-Month Default Flag"
SEED = 42
N_SPLITS = 5
PRED_DIR = PREDS
MODELS = ["catboost", "lgbm"]

y = pd.read_csv(DATA / "train_dataset.csv", usecols=[TARGET])[TARGET]
test_ids = pd.read_csv(DATA / "test_without_gt.csv", usecols=["ID"])["ID"]

oofs, tests = {}, {}
for name in MODELS:
    if not (PRED_DIR / f"{name}_oof.npy").exists():
        print(f"Skipping {name}: run its notebook first")
        continue
    oofs[name] = np.load(PRED_DIR / f"{name}_oof.npy")
    tests[name] = np.load(PRED_DIR / f"{name}_test.npy")
    assert len(oofs[name]) == len(y) and len(tests[name]) == len(test_ids)

print("Loaded:", list(oofs))

Loaded: ['catboost_tuned', 'lgbm', 'logreg']


## 3. Score each model on the same folds

We rebuild the same 5 folds and score every model on each one. This gives a fair, side-by-side comparison, and lets you see whether one model beats another **consistently** or just on average.

In [88]:
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
folds = [val_idx for _, val_idx in cv.split(np.zeros(len(y)), y)]

def fold_scores(pred):
    return [roc_auc_score(y.iloc[idx], pred[idx]) for idx in folds]

table = pd.DataFrame({name: fold_scores(p) for name, p in oofs.items()})
table.index = [f"fold {i}" for i in range(N_SPLITS)]
table.loc["mean"] = table.mean()
table.round(4)

,catboost_tuned,lgbm,logreg
fold 0,0.6504,0.6488,0.6304
fold 1,0.6646,0.6693,0.6436
fold 2,0.6477,0.6456,0.6366
fold 3,0.6593,0.6571,0.6359
fold 4,0.6384,0.6391,0.6321
mean,0.6521,0.6520,0.6357


## 4. How similar are the models?

This is the **correlation** of the models' rank orderings, from -1 to 1. A value near 1 means two models rank people almost identically, so blending them adds little. Lower values mean they disagree more, which gives blending more to work with.

Disagreement only helps if both models are reasonably good. A model that disagrees because it's bad will drag the blend down.

In [89]:
ranks_oof = pd.DataFrame({name: rankdata(p) for name, p in oofs.items()})
ranks_oof.corr().round(3)

,catboost_tuned,lgbm,logreg
catboost_tuned,1.000,0.934,0.862
lgbm,0.934,1.000,0.841
logreg,0.862,0.841,1.000


## 5. Try every combination

For each group of 2 or more models, we average their **ranks** and score the result.

**Why ranks:** each model's scores sit on a different scale. Logistic regression might spread its scores out more than CatBoost does, so a plain average would let one model dominate. Ranks put them on the same scale, and AUC only cares about order, so nothing is lost.

`wins` counts how many folds the blend beats the best single model on. A blend that wins only on average, but not in most folds, may not be a real gain.

In [90]:
def blend(preds):
    return np.mean([rankdata(p) for p in preds], axis=0)

best_single = table.loc["mean"].idxmax()
best_folds = table.loc[table.index != "mean", best_single].values

rows = []
for k in range(2, len(oofs) + 1):
    for combo in combinations(oofs, k):
        s = fold_scores(blend([oofs[n] for n in combo]))
        rows.append({
            "models": " + ".join(combo),
            "cv_auc": np.mean(s),
            "std": np.std(s),
            f"wins vs {best_single}": int((np.array(s) > best_folds).sum()),
        })

print(f"Best single model: {best_single}, CV AUC {table.loc['mean', best_single]:.4f}")
results = pd.DataFrame(rows).sort_values("cv_auc", ascending=False)
results.round(4)

Best single model: catboost_tuned, CV AUC 0.6521


,models,cv_auc,std,wins vs catboost_tuned
0,catboost_tuned + lgbm,0.6543,0.0099,5
3,catboost_tuned + lgbm + logreg,0.6524,0.0080,3
2,lgbm + logreg,0.6494,0.0072,1
1,catboost_tuned + logreg,0.6488,0.0067,2


## 6. Choose the blend

Set `CHOSEN` to the models you want to combine. A good choice:
- has the highest or near-highest CV AUC,
- beats the best single model in most folds, and
- uses equal weights, which we're already doing. Tuned weights can fit noise in these predictions and then disappoint on the private leaderboard.

If no blend clearly beats your best single model, submit the single model. That's a perfectly good result.

In [91]:
CHOSEN = list(oofs)  # for example ["catboost", "lgbm"]

blend_oof = blend([oofs[n] for n in CHOSEN])
print("Chosen:", CHOSEN)
print(f"Blend CV AUC: {np.mean(fold_scores(blend_oof)):.4f}")

Chosen: ['catboost_tuned', 'lgbm', 'logreg']
Blend CV AUC: 0.6524


## 7. Write the blended submission

The same rank averaging, applied to the test predictions. Ranks run from 1 to the number of test rows, so dividing by that number puts them between 0 and 1, as the competition requires.

These values aren't true probabilities anymore, just a risk ordering. That's fine, because AUC only uses the order.

In [92]:
test_pred = blend([tests[n] for n in CHOSEN]) / len(test_ids)

sub = pd.read_csv(DATA / "submission_template_for_test.csv")
assert (sub["ID"].values == test_ids.values).all(), "Row order doesn't match the template"

sub[TARGET] = test_pred

assert list(sub.columns) == ["ID", TARGET]
assert len(sub) == 137_091
assert sub[TARGET].between(0, 1).all() and sub[TARGET].notna().all()

name = "submission_blend_" + "_".join(CHOSEN) + ".csv"
sub.to_csv(SUBMISSIONS / name, index=False)
print("Saved submissions/" + name)

Saved submission_blend_catboost_tuned_lgbm_logreg.csv


## Your turn

1. Which single model is best, and does it win in every fold?
2. Which pair of models is least correlated? Does that pair also blend best?
3. Does adding logistic regression help the tree blend, or hurt it?
4. Which two submissions would you pick as your final two, and why?